# 策略对比: max(0,6) vs max(2,6)

## 两种周末报价策略的累计PnL对比

- **策略1 (max(0,6))**: 用 `max(hour_0, hour_6)` 的USDCNH作为周末报价
- **策略2 (max(2,6))**: 用 `max(hour_2, hour_6)` 的USDCNH作为周末报价

### 交易量
- USD: 60M USD/周
- JPY: 4B JPY/周

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.font_manager import FontProperties
import warnings
warnings.filterwarnings('ignore')

# ======== 中文字体设置 (Windows) ========
# 直接从字体文件加载
font_path = r'C:\Windows\Fonts\msyh.ttc'  # 微软雅黑
if not os.path.exists(font_path):
    font_path = r'C:\Windows\Fonts\simhei.ttf'  # 黑体
if not os.path.exists(font_path):
    font_path = r'C:\Windows\Fonts\simsun.ttc'  # 宋体

chinese_font = FontProperties(fname=font_path, size=12)
chinese_font_title = FontProperties(fname=font_path, size=14)
chinese_font_legend = FontProperties(fname=font_path, size=11)
print(f'Using font: {font_path}')

OUTPUT_DIR = r'c:\Users\tencentren\CodeBuddy\FX_SYSTEM\bmad-quant-system\output'

# 交易量
USD_WEEKLY_VOLUME = 60_000_000      # 6000万 USD/周
JPY_WEEKLY_VOLUME = 4_000_000_000   # 40亿 JPY/周

print('\n交易量设置:')
print(f'  USD: {USD_WEEKLY_VOLUME/1e6:.0f}M USD/周')
print(f'  JPY: {JPY_WEEKLY_VOLUME/1e9:.0f}B JPY/周')

## 1. 加载数据

In [ ]:
# 加载USDCNH和JPYCNH数据
all_files = os.listdir(OUTPUT_DIR)

# USDCNH 1年数据
usdcnh_files = [f for f in all_files if 'USDCNH' in f and '1year' in f and f.endswith('.xlsx') and not f.startswith('~$')]
print(f"Loading USDCNH: {usdcnh_files[0]}")
df_usdcnh = pd.read_excel(os.path.join(OUTPUT_DIR, usdcnh_files[0]))
df_usdcnh['timestamp'] = pd.to_datetime(df_usdcnh['timestamp'])
df_usdcnh.set_index('timestamp', inplace=True)

# JPYCNH 1年数据
jpycnh_files = [f for f in all_files if 'JPYCNH' in f and '1year' in f and f.endswith('.xlsx') and not f.startswith('~$')]
print(f"Loading JPYCNH: {jpycnh_files[0]}")
df_jpycnh = pd.read_excel(os.path.join(OUTPUT_DIR, jpycnh_files[0]))
df_jpycnh['timestamp'] = pd.to_datetime(df_jpycnh['timestamp'])
df_jpycnh.set_index('timestamp', inplace=True)

print(f"\nUSDCNH: {len(df_usdcnh)} rows, {df_usdcnh.index.min()} ~ {df_usdcnh.index.max()}")
print(f"JPYCNH: {len(df_jpycnh)} rows, {df_jpycnh.index.min()} ~ {df_jpycnh.index.max()}")

In [ ]:
# 准备周六数据 (0-6点)
for df in [df_usdcnh, df_jpycnh]:
    df['weekday'] = df.index.dayofweek
    df['date'] = df.index.date
    df['hour'] = df.index.hour
    if 'mid' not in df.columns:
        df['mid'] = (df['bid'] + df['ask']) / 2

usdcnh_sat = df_usdcnh[(df_usdcnh['weekday'] == 5) & (df_usdcnh['hour'] <= 6)].copy()
jpycnh_sat = df_jpycnh[(df_jpycnh['weekday'] == 5) & (df_jpycnh['hour'] <= 6)].copy()

# 共同日期
common_dates = sorted(set(usdcnh_sat['date'].unique()) & set(jpycnh_sat['date'].unique()))
print(f"Common Saturday dates: {len(common_dates)} weeks")
print(f"Date range: {common_dates[0]} ~ {common_dates[-1]}")

## 2. 计算两种策略的PnL

In [ ]:
# 计算每周的策略PnL
weekly_data = []

for sat_date in common_dates:
    usdcnh_day = usdcnh_sat[usdcnh_sat['date'] == sat_date]
    jpycnh_day = jpycnh_sat[jpycnh_sat['date'] == sat_date]
    
    if usdcnh_day.empty or jpycnh_day.empty:
        continue
    
    # 获取各小时USDCNH价格
    usdcnh_h0 = usdcnh_day[usdcnh_day['hour'] == 0]['mid'].mean() if len(usdcnh_day[usdcnh_day['hour'] == 0]) > 0 else np.nan
    usdcnh_h2 = usdcnh_day[usdcnh_day['hour'] == 2]['mid'].mean() if len(usdcnh_day[usdcnh_day['hour'] == 2]) > 0 else np.nan
    usdcnh_h6 = usdcnh_day[usdcnh_day['hour'] == 6]['mid'].mean() if len(usdcnh_day[usdcnh_day['hour'] == 6]) > 0 else np.nan
    
    # 获取JPYCNH价格 (每100日元)
    jpycnh_avg = jpycnh_day['mid'].mean()  # 这是每100日元的价格
    jpycnh_per_jpy = jpycnh_avg / 100      # 每1日元的价格
    
    if pd.isna(usdcnh_h0) or pd.isna(usdcnh_h2) or pd.isna(usdcnh_h6):
        continue
    
    # 两种策略的参照物
    ref_max_0_6 = max(usdcnh_h0, usdcnh_h6)  # 策略1: max(0,6)
    ref_max_2_6 = max(usdcnh_h2, usdcnh_h6)  # 策略2: max(2,6)
    
    # ========== USD PnL ==========
    usd_pnl_diff = (ref_max_0_6 - ref_max_2_6) * USD_WEEKLY_VOLUME
    
    # ========== JPY PnL ==========
    usdcnh_avg = usdcnh_day['mid'].mean()
    jpycnh_for_max_0_6 = jpycnh_per_jpy * (ref_max_0_6 / usdcnh_avg)
    jpycnh_for_max_2_6 = jpycnh_per_jpy * (ref_max_2_6 / usdcnh_avg)
    jpy_pnl_diff = (jpycnh_for_max_0_6 - jpycnh_for_max_2_6) * JPY_WEEKLY_VOLUME
    
    weekly_data.append({
        'date': sat_date,
        'ref_max_0_6': ref_max_0_6,
        'ref_max_2_6': ref_max_2_6,
        'usd_pnl_diff': usd_pnl_diff,
        'jpy_pnl_diff': jpy_pnl_diff,
        'total_pnl_diff': usd_pnl_diff + jpy_pnl_diff,
    })

df = pd.DataFrame(weekly_data)
print(f"Calculation complete: {len(df)} weeks")

In [ ]:
# 计算累计PnL差异
df['usd_cum_diff'] = df['usd_pnl_diff'].cumsum()
df['jpy_cum_diff'] = df['jpy_pnl_diff'].cumsum()
df['total_cum_diff'] = df['total_pnl_diff'].cumsum()

print("Cumulative PnL Difference (max(0,6) - max(2,6)):")
print(f"  USD:   {df['usd_cum_diff'].iloc[-1]:,.0f} CNH ({df['usd_cum_diff'].iloc[-1]/1e6:.2f}M)")
print(f"  JPY:   {df['jpy_cum_diff'].iloc[-1]:,.0f} CNH ({df['jpy_cum_diff'].iloc[-1]/1e6:.2f}M)")
print(f"  Total: {df['total_cum_diff'].iloc[-1]:,.0f} CNH ({df['total_cum_diff'].iloc[-1]/1e6:.2f}M)")

## 3. 绘制累计PnL差异线图

In [ ]:
# 绘制累计PnL差异线图
fig, ax = plt.subplots(figsize=(14, 8))

dates = pd.to_datetime(df['date'])

# 绘制三条线
ax.plot(dates, df['usd_cum_diff']/1e6, 'b-', linewidth=2.5, marker='o', markersize=6)
ax.plot(dates, df['jpy_cum_diff']/1e6, 'g-', linewidth=2.5, marker='s', markersize=6)
ax.plot(dates, df['total_cum_diff']/1e6, 'r-', linewidth=3, marker='^', markersize=8)

# 零线
ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.7)

# 填充区域
ax.fill_between(dates, 0, df['total_cum_diff']/1e6, 
                where=(df['total_cum_diff'] >= 0), color='green', alpha=0.15)
ax.fill_between(dates, 0, df['total_cum_diff']/1e6, 
                where=(df['total_cum_diff'] < 0), color='red', alpha=0.15)

# 标注最终值
final_usd = df['usd_cum_diff'].iloc[-1]/1e6
final_jpy = df['jpy_cum_diff'].iloc[-1]/1e6
final_total = df['total_cum_diff'].iloc[-1]/1e6

# 使用fontproperties参数设置中文字体
ax.annotate(f'美金: {final_usd:.2f}M', xy=(dates.iloc[-1], final_usd), xytext=(10, 0),
            textcoords='offset points', fontsize=11, color='blue', fontweight='bold',
            fontproperties=chinese_font)
ax.annotate(f'日元: {final_jpy:.2f}M', xy=(dates.iloc[-1], final_jpy), xytext=(10, 0),
            textcoords='offset points', fontsize=11, color='green', fontweight='bold',
            fontproperties=chinese_font)
ax.annotate(f'合计: {final_total:.2f}M', xy=(dates.iloc[-1], final_total), xytext=(10, 5),
            textcoords='offset points', fontsize=12, color='red', fontweight='bold',
            fontproperties=chinese_font,
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

ax.set_xlabel('日期', fontproperties=chinese_font)
ax.set_ylabel('累计PnL差异 (百万 CNH)', fontproperties=chinese_font)
ax.set_title('累计PnL对比: max(0,6) - max(2,6)\n(正值表示 max(0,6) 策略更优)', 
             fontproperties=chinese_font_title)

# 中文图例
legend_labels = ['美金 (6000万/周)', '日元 (40亿/周)', '合计 (美金+日元)']
ax.legend(legend_labels, loc='upper left', prop=chinese_font_legend)

ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cumulative_pnl_usd_jpy_total.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. 详细统计

In [ ]:
print("="*70)
print("Strategy Comparison: max(0,6) vs max(2,6)")
print("="*70)

print(f"\nPeriod: {df['date'].iloc[0]} ~ {df['date'].iloc[-1]}")
print(f"Total weeks: {len(df)}")

print(f"\nVolume:")
print(f"  USD: {USD_WEEKLY_VOLUME/1e6:.0f}M USD/week")
print(f"  JPY: {JPY_WEEKLY_VOLUME/1e9:.0f}B JPY/week")

print(f"\n" + "-"*70)
print("Cumulative PnL Difference [max(0,6) - max(2,6)]:")
print("-"*70)
print(f"{'Currency':>10} {'Total Diff':>20} {'Weekly Avg':>20} {'Annual (52wk)':>20}")
print("-"*70)

for name, cum_col, diff_col in [('USD', 'usd_cum_diff', 'usd_pnl_diff'),
                                 ('JPY', 'jpy_cum_diff', 'jpy_pnl_diff'),
                                 ('Total', 'total_cum_diff', 'total_pnl_diff')]:
    total = df[cum_col].iloc[-1]
    weekly = df[diff_col].mean()
    annual = weekly * 52
    print(f"{name:>10} {total:>15,.0f} CNH {weekly:>15,.0f} CNH {annual:>15,.0f} CNH")

print("-"*70)

# Win/Loss stats
print(f"\nWin/Loss Stats:")
print(f"  max(0,6) wins: {(df['total_pnl_diff'] > 0).sum()} weeks ({(df['total_pnl_diff'] > 0).sum()/len(df)*100:.1f}%)")
print(f"  max(2,6) wins: {(df['total_pnl_diff'] < 0).sum()} weeks ({(df['total_pnl_diff'] < 0).sum()/len(df)*100:.1f}%)")
print(f"  Tie: {(df['total_pnl_diff'] == 0).sum()} weeks ({(df['total_pnl_diff'] == 0).sum()/len(df)*100:.1f}%)")

print(f"\nConclusion: {'max(0,6) is BETTER!' if df['total_cum_diff'].iloc[-1] > 0 else 'max(2,6) is BETTER!'}")
print(f"  Extra revenue: {abs(df['total_cum_diff'].iloc[-1]):,.0f} CNH ({abs(df['total_cum_diff'].iloc[-1])/1e6:.2f}M)")

## 5. 每周PnL差异明细

In [ ]:
# 显示每周明细
display_df = df[['date', 'ref_max_0_6', 'ref_max_2_6', 
                 'usd_pnl_diff', 'jpy_pnl_diff', 'total_pnl_diff',
                 'usd_cum_diff', 'jpy_cum_diff', 'total_cum_diff']].copy()

# 格式化
for col in ['usd_pnl_diff', 'jpy_pnl_diff', 'total_pnl_diff', 
            'usd_cum_diff', 'jpy_cum_diff', 'total_cum_diff']:
    display_df[col] = display_df[col].apply(lambda x: f"{x:,.0f}")

display_df.columns = ['Date', 'Ref max(0,6)', 'Ref max(2,6)',
                      'USD Diff', 'JPY Diff', 'Total Diff',
                      'USD Cum', 'JPY Cum', 'Total Cum']

display(display_df)

In [ ]:
# 保存结果
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
save_path = os.path.join(OUTPUT_DIR, f'strategy_comparison_usd_jpy_{timestamp}.xlsx')
df.to_excel(save_path, index=False)
print(f"Results saved: {save_path}")